# LifeMate run (Colab)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
# Option A: unzip the code package uploaded to Drive (zip from ops/scripts/pack_code.sh).
!mkdir -p /content/lifemate && unzip -oq "$(ls -t /content/drive/MyDrive/zac2026/lifemate-code-*.zip | head -1)" -d /content/lifemate
# Option B: !git clone <repo-url> /content/lifemate
%cd /content/lifemate

In [ ]:
%pip install -q -r ops/requirements/notebook.txt pytest
# vllm version pinned from uv.lock at authoring time (0.32.0); only on GPU machines.
!nvidia-smi > /dev/null 2>&1 && pip install -q vllm==0.32.0 || echo "no GPU: skipping vllm"
# faster-whisper (ASR mặc định) nằm trong extra gpu cùng vLLM nên không có trong notebook.txt.
!pip install -q faster-whisper==1.2.1


In [ ]:
!python -c "from pathlib import Path; from lifemate.core.envinfo import detect; print(detect(Path('.')))"

In [ ]:
# Danh sách split không có trên git: dựng lại (giống từng byte) nếu thiếu.
!test -f configs/splits/train.txt || python -m lifemate.eval.make_splits --env colab --smoke 30


In [ ]:
!python -m pytest -q -m "not data and not tokenizer"

In [ ]:
!python -m lifemate.run.cli --config configs/v0.toml --env colab --split train --ids configs/splits/smoke.txt --name smoke

In [ ]:
# Baseline: transcript ASR thật (`--transcripts asr`); v0.toml để gold cho vòng lặp prompt.
# GPU cần từ 24 GB (L4, A30, A100): T4 15 GB không đủ cho vLLM + whisper-large-v3 ở dtype auto.
TAG = "base"
!python -m lifemate.run.cli --config configs/v0.toml --env colab --split train --ids configs/splits/val_seen.txt --transcripts asr --name {TAG}_val_seen_asr
!python -m lifemate.run.cli --config configs/v0.toml --env colab --split train --ids configs/splits/val_unseen.txt --transcripts asr --name {TAG}_val_unseen_asr
